# gerbil-data: Feature Engineering Pipeline for Recommender Systems

This notebook demonstrates the end-to-end workflow of gerbil-data using the MovieLens 1M dataset.

## Overview

1. **Download raw data** — Fetch ML-1M from GroupLens
2. **Run ETL pipeline** — Clean, extract features, join into wide table
3. **Run featurization** — Build vocabulary, encode to TFRecord
4. **Inspect output** — Verify TFRecord format and feature values
5. **Train a model** — Quick example using the generated features

## Prerequisites

Make sure you have built the project and downloaded the dataset:

```bash
mvn clean package -DskipTests
curl -O https://files.grouplens.org/datasets/movielens/ml-1m.zip
unzip ml-1m.zip -d /tmp/ml-1m
```

---
## Step 1: Raw Data Inspection

Let's look at the raw MovieLens 1M data before any processing.

In [ ]:
import pandas as pd
import os

DATA_DIR = "/tmp/ml-1m"

# Ratings
ratings = pd.read_csv(
    os.path.join(DATA_DIR, "ratings.dat"),
    sep="::",
    names=["UserID", "MovieID", "Rating", "Timestamp"],
    engine="python",
    encoding="latin-1"
)
print(f"Ratings: {ratings.shape[0]} rows")
ratings.head()

In [ ]:
# Users
users = pd.read_csv(
    os.path.join(DATA_DIR, "users.dat"),
    sep="::",
    names=["UserID", "Gender", "Age", "Occupation", "ZipCode"],
    engine="python",
    encoding="latin-1"
)
print(f"Users: {users.shape[0]} rows")
users.head()

In [ ]:
# Movies
movies = pd.read_csv(
    os.path.join(DATA_DIR, "movies.dat"),
    sep="::",
    names=["MovieID", "Title", "Genres"],
    engine="python",
    encoding="latin-1"
)
print(f"Movies: {movies.shape[0]} rows")
movies.head()

---
## Step 2: Run the ETL Pipeline via Spark

The ETL pipeline transforms raw data into a joined feature table with 4 stages:

1. **ML1MCleanSample** — Parse and clean raw ratings, filter invalid records
2. **ML1MMovieStatFeature** — Compute movie-level statistics (rating count, avg rating, hot rank)
3. **ML1MUserMovieRateSequence** — Build user behavior sequences by time window
4. **ML1MJoinSample** — Join all features into one wide table

Each stage outputs to a subdirectory and runs quality checks automatically.

In [ ]:
%%bash
# Run the pipeline using shell scripts
JAR="target/gerbil-data-1.0.0-jar-with-dependencies.jar"
ML1M_HOME="/tmp/ml-1m"
OUTPUT="/tmp/gerbil-output"

spark-submit --class processing.clean.ML1MCleanSample $JAR $ML1M_HOME
spark-submit --class processing.feature.ML1MMovieStatFeature $JAR $ML1M_HOME
spark-submit --class processing.feature.ML1MUserMovieRateSequence $JAR $ML1M_HOME
spark-submit --class processing.join.ML1MJoinSample $JAR $ML1M_HOME
echo "ETL complete"

---
## Step 3: Inspect the Joined Feature Table

After the ETL pipeline, the join_sample contains all features as a flat table with JSON-encoded feature columns.

In [ ]:
import json

join_path = os.path.join(ML1M_HOME, "join_sample")
if os.path.exists(join_path):
    join_df = pd.read_csv(
        os.path.join(join_path, os.listdir(join_path)[0]),
        sep="\t",
        names=["user_id", "item_id", "timestamp", "rating", "day",
               "user_profile", "movie_feature", "user_behavior"],
        nrows=5
    )
    print(f"Join sample columns: {list(join_df.columns)}")
    
    # Parse a user_profile JSON to show extracted features
    sample_profile = json.loads(join_df.iloc[0]["user_profile"])
    print("\nExample user_profile features:")
    for k, v in sample_profile.items():
        print(f"  {k}: {v}")
        
    sample_item = json.loads(join_df.iloc[0]["movie_feature"])
    print("\nExample movie_feature:")
    for k, v in sample_item.items():
        print(f"  {k}: {v}")
else:
    print("Join sample not found. Run the ETL pipeline first.")

---
## Step 4: Feature Encoding (Featurization)

The `ML1MPipeline` takes the joined data and:
1. Parses samples into `ML1MSample` objects
2. Computes feature hashes and builds a vocabulary (pos-map)
3. Encodes each sample into TFRecord Example protobuf

Key encoding format: each feature produces `{name}_raw / _index / _value` triplets:
- `_raw`: original string value
- `_index`: embedding position (hashed or vocabulary lookup)
- `_value`: feature weight (1.0 for categorical, actual value for continuous)

In [ ]:
%%bash
spark-submit --class pipeline.ML1MPipeline \
  --conf spark.serializer=org.apache.spark.serializer.JavaSerializer \
  target/gerbil-data-1.0.0-jar-with-dependencies.jar \
  --yesterday 20010101 \
  --parts 1 \
  --feature_threshold 5 \
  --target_threshold 5 \
  --sample_ratio 0.1 \
  --input_dir ${ML1M_HOME} \
  --output_dir ${OUTPUT} \
  --output_format tfrecord
echo "Featurization complete"

---
## Step 5: Inspect TFRecord Output

The pipeline outputs TFRecord files containing TensorFlow Example protobufs.
Each Example contains all features as `{name}_raw`, `{name}_index`, `{name}_value` tensors.

In [ ]:
import tensorflow as tf
import numpy as np

def inspect_tfrecord(path):
    """Read and display a TFRecord file's structure."""
    dataset = tf.data.TFRecordDataset(path)
    count = 0
    for raw_record in dataset.take(3):
        example = tf.train.Example()
        example.ParseFromString(raw_record.numpy())
        features = example.features.feature
        print(f"\n--- Record {count} ---")
        print(f"Number of feature fields: {len(features)}")
        
        # Show first 10 feature names
        feat_names = sorted(features.keys())[:10]
        for name in feat_names:
            feat = features[name]
            kind = feat.WhichOneof('kind')
            if kind == 'bytes_list':
                vals = [v.decode() for v in feat.bytes_list.value[:3]]
            elif kind == 'int64_list':
                vals = list(feat.int64_list.value[:3])
            elif kind == 'float_list':
                vals = list(feat.float_list.value[:3])
            print(f"  {name}: {kind}, values={vals}")
        count += 1
    return count

tfrecord_dir = os.path.join(OUTPUT, "20010101", "tfrecord")
if os.path.exists(tfrecord_dir):
    files = [f for f in os.listdir(tfrecord_dir) if f.endswith(".tfrecord")]
    if files:
        inspect_tfrecord(os.path.join(tfrecord_dir, files[0]))
    else:
        print("No .tfrecord files found. Check pipeline output.")
else:
    print("TFRecord output not found. Run the featurization pipeline first.")

---
## Step 6: Vocabulary Files

The pipeline also produces vocabulary files that map feature values to embedding positions.

In [ ]:
import json

vocab_path = os.path.join(OUTPUT, "20010101", "pos_map.json")
if os.path.exists(vocab_path):
    with open(vocab_path) as f:
        vocab = json.load(f)
    print(f"Vocabulary entries: {len(vocab)}")
    print("\nSample entries:")
    for i, entry in enumerate(vocab[:5]):
        print(f"  {entry}")

---
## Step 7: Quick Model Training

With the generated TFRecord data, you can directly train a deep learning model.
Here's a minimal example using TensorFlow's Keras API.

In [ ]:
import tensorflow as tf

def parse_tfrecord(proto):
    """Parse a TFRecord Example into feature dict and label."""
    feature_spec = {
        "target": tf.io.FixedLenFeature([], tf.float32),
        "user_age_index": tf.io.VarLenFeature(tf.int64),
        "user_gender_index": tf.io.VarLenFeature(tf.int64),
        "movie_title_index": tf.io.VarLenFeature(tf.int64),
        "movie_genres_index": tf.io.VarLenFeature(tf.int64),
        "context_time_hour_index": tf.io.VarLenFeature(tf.int64),
        # Add more features as needed
    }
    parsed = tf.io.parse_single_example(proto, feature_spec)
    label = parsed.pop("target")
    return parsed, label

# Load dataset (if TFRecord files exist)
tfrecord_dir = os.path.join(OUTPUT, "20010101", "tfrecord")
if os.path.exists(tfrecord_dir):
    files = [os.path.join(tfrecord_dir, f) for f in os.listdir(tfrecord_dir) if f.endswith(".tfrecord")]
    if files:
        dataset = tf.data.TFRecordDataset(files).map(parse_tfrecord).batch(32)
        print(f"Dataset ready with {len(files)} file(s)")
        print("Features:", list(dataset.element_spec[0].keys()))
    else:
        print("No TFRecord files found.")
else:
    print("TFRecord directory not found.")

---
## Summary

This notebook demonstrated:

1. **Raw data inspection** — Understanding the ML-1M dataset
2. **ETL processing** — Cleaning, feature extraction, and multi-table joining via Spark
3. **Featurization** — Converting raw features to embedding-compatible `{name}_raw / _index / _value` triplets
4. **TFRecord output** — Production-ready training data for TensorFlow models
5. **Vocabulary management** — Feature position maps for consistent offline/online serving

For more details, see the [project README](../README.md) and [source code](../src/main/scala/).

---
*Notebook generated with gerbil-data $\heartsuit$*